In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from catboost import Pool, CatBoostClassifier
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, log_loss
from sklearn.impute import SimpleImputer

In [159]:
dico={
    "cat_1":["NEFT", "RTGS", "IMPS", "RMCPL"],
    "cat_2":["CASHDEP", "ICA17825"],
    "cat_3":["POSACQ", "CHQDEPOSITRETURN"],
    "cat_4":["CHQDEP/OWDEL1"],
    "cat_5":["AEPS/TAX", "AEPS/FEES", "CHARGES"],
    "cat_6":["BEAT", "DSB"],
    "cat_7":["INCOME"],
    "cat_8":["BBPSDT", "BBPSSE"],
    "cat_9":["NAAPTOLONLINESHOPPINGP", "STAMPPAPERPURCHASING", "PAYMENTSFOR"],
    "cat_10":["VODAFONE", "AIRTEL"],
    "cat_11":["INT.COLL"],
    "cat_12":["LOANRECOVERYFOR"],
    "cat_13":["REV", "IWCHQRETURN"],
    "cat_14":["TIMESINTERNETLIMITED"],
    "cat_15":["INSURANCE"],
    "cat_16":["RAIASSOCIATES"],
    "cat_17":["BOOKING"],
    "cat_18":["NETHOPE"],
}

In [160]:
def mapping(x):
    for i in dico.keys():
        for j in dico[i]:
            y=j.split("/")
            if len(y)==2:
                if ((y[0] in x)+(y[1] in x))==2:
                    return i
            if len(y)==1:
                if y[0] in x:
                    return i
          
    return x

In [172]:
def modif(df):
    df["Account_NO"]=df["Account_NO"].map(str)
    df["year"]=df.DATE.map(lambda x: x.split("-")[0])
    df["month"]=df.DATE.map(lambda x: x.split("-")[1])
    df["var_1"]=df["TRANSACTION DETAILS"].map(mapping)
    df.drop(["DATE","month","TRANSACTION DETAILS"],axis=1, inplace=True)
    if "ID" in df.columns:
        df.drop(["ID","CHQ.NO."],axis=1, inplace=True)
    return df

In [173]:
train = pd.read_csv("Train.csv")

# Filter train dataset
train = train[train.Category.notnull()]
train.drop("CHQ.NO.",axis=1, inplace=True)
train.dropna(axis=0,inplace=True)
val_columns=[i for i in train.Category.unique() if (train.Category[train.Category==i].shape[0]/train.shape[0])>0.001]
train=train[train.Category.isin(val_columns)]

# Feature Engenireering
train=modif(train)
target_enc=LabelEncoder()
train.Category=target_enc.fit_transform(train.Category)


In [174]:
train.head()

,Account_NO,WITHDRAWAL AMT,DEPOSIT AMT,BALANCE AMT,Category,year,var_1
0,84903292,0.0,15000000.0,15000000.0,9,2014,cat_1
1,84903292,0.0,15000000.0,60000000.0,9,2014,cat_1
2,84903292,0.0,15000000.0,90000000.0,9,2014,cat_1
3,84903292,0.0,10000000.0,100000000.0,9,2014,cat_1
4,84903292,0.0,15000000.0,45000000.0,9,2014,cat_1


In [191]:
X_train, X_test, y_train, y_test=train_test_split(train.drop(["Category"], axis=1), train.Category, test_size=.6 ,random_state=25)

In [192]:
cat_features=X_train.select_dtypes("object").columns.values

train_data=Pool(
    data=X_train,
    label=y_train,
    cat_features=cat_features
)
eval_data=Pool(
    data=X_test,
    label=y_test,
    cat_features=cat_features
)

In [199]:
model = CatBoostClassifier(iterations=150,
                           learning_rate=0.17,
                           depth=8,
                           custom_metric=['MultiClass']
)
# Fit model
model.fit(
    train_data,
    eval_set=eval_data,
    verbose=True
)
# Get predicted classes
preds_class = model.predict(eval_data)

0:	learn: 1.2590838	test: 1.2518911	best: 1.2518911 (0)	total: 824ms	remaining: 2m 2s
1:	learn: 1.0122121	test: 1.0064227	best: 1.0064227 (1)	total: 1.67s	remaining: 2m 3s
2:	learn: 0.8608255	test: 0.8562301	best: 0.8562301 (2)	total: 2.41s	remaining: 1m 58s
3:	learn: 0.7433977	test: 0.7392987	best: 0.7392987 (3)	total: 3.13s	remaining: 1m 54s
4:	learn: 0.6525949	test: 0.6502941	best: 0.6502941 (4)	total: 4.17s	remaining: 2m
5:	learn: 0.5844165	test: 0.5839038	best: 0.5839038 (5)	total: 4.92s	remaining: 1m 58s
6:	learn: 0.5306916	test: 0.5316969	best: 0.5316969 (6)	total: 5.7s	remaining: 1m 56s
7:	learn: 0.4136947	test: 0.4125356	best: 0.4125356 (7)	total: 6.45s	remaining: 1m 54s
8:	learn: 0.3392178	test: 0.3366286	best: 0.3366286 (8)	total: 7.18s	remaining: 1m 52s
9:	learn: 0.2842109	test: 0.2809049	best: 0.2809049 (9)	total: 7.92s	remaining: 1m 50s
10:	learn: 0.2418066	test: 0.2376872	best: 0.2376872 (10)	total: 8.65s	remaining: 1m 49s
11:	learn: 0.2070345	test: 0.2024171	best: 0.202

In [200]:
pd.DataFrame(model.feature_importances_, index=X_train.columns.values, columns=["Importance"])

,Importance
Account_NO,25.971172
WITHDRAWAL AMT,5.790612
DEPOSIT AMT,22.348345
BALANCE AMT,5.551536
year,19.114955
var_1,21.223380


In [201]:
# Feature Engenireering
test=pd.read_csv("Test.csv")
test=modif(test)

prediction=pd.DataFrame(model.predict_proba(test),columns=target_enc.inverse_transform(model.classes_))

In [202]:
ss=pd.read_csv("SampleSubmission.csv")

for column in prediction.columns:
    ss[column]=prediction[column]

In [203]:
ss.head()

,ID,Bank Charges,Bill-Payments,Cash-Pickup,Cell Phone and Airtime,Cheque-Payment,Deposit,Donations,General Purchases,Insurance,...,Internet and IT Services,Loan Repayment,Merchant-Payment,Money-Transfer,Professional services,Reversal,Salary and wages,Savings and Investments,Shopping,"Transport, Travel, and Logistics"
0,2014-02-28 00:00:00X3000000000,0.000909,0.000304,0.000304,0.000205,0.000554,0.000483,0,0,0,...,0,0.000275,0.000878,0.978089,0,0.014344,0.003201,0,0.000258,0
1,2014-02-28 00:00:00X7500000000,0.000970,0.000302,0.000302,0.000204,0.000535,0.000471,0,0,0,...,0,0.000271,0.000874,0.977251,0,0.014958,0.003402,0,0.000269,0
2,2014-03-08 00:00:00X4500002863,0.245525,0.000703,0.000601,0.000573,0.001321,0.000779,0,0,0,...,0,0.000621,0.001654,0.671932,0,0.032421,0.042722,0,0.000682,0
3,2014-03-08 00:00:00X6000002863,0.011745,0.001272,0.000509,0.000800,0.001872,0.000643,0,0,0,...,0,0.001273,0.001143,0.958071,0,0.014019,0.006797,0,0.000864,0
4,2014-03-08 00:00:00X6000005726,0.245525,0.000703,0.000601,0.000573,0.001321,0.000779,0,0,0,...,0,0.000621,0.001654,0.671932,0,0.032421,0.042722,0,0.000682,0


In [ ]:
ss.to_csv("submission_42.csv", index=False)